In [1]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

In [2]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

In [3]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .master("local[*]")
    .appName("season_sanity_events")
    .getOrCreate()
    )

In [4]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events_2022_2023 = spark.read.parquet(*season_events_parquet_file_paths)

df_events_2022_2023.show(5)

+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+-----------+--------------+-----------------------+-----------------------+
|             eventId|competitionId|gameId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|      details_parsed|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|eventPlayerId| eventPlayerName|eventTeamId| eventTeamName|eventSubTypeDescription|eventOutcomeDescription|
+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+-----------+-----------

In [5]:
print(f'Quantidade total de eventos: {df_events_2022_2023.select('eventId').count()}')
print(f'Quantidade de eventos distintos: {df_events_2022_2023.select('eventId').distinct().count()}')

print(f'Quantidade de eventos duplicados: {df_events_2022_2023.select('eventId').count() - df_events_2022_2023.select('eventId').distinct().count()}')

Quantidade total de eventos: 945154
Quantidade de eventos distintos: 944838
Quantidade de eventos duplicados: 316


- Pode-se notar que nessa temporada também há eventos duplicados e é importante removê-los ao inicio do pipeline de engenharia dos dados.

In [6]:
(
    df_events_2022_2023
    .groupBy('gameId')
    .agg(F.count(F.col('eventId')).alias('qtd_eventos'))
    .agg(F.round(F.avg(F.col('qtd_eventos')), 2).alias('Quantidade média de eventos por partida'))
).show()

+---------------------------------------+
|Quantidade média de eventos por partida|
+---------------------------------------+
|                                2487.25|
+---------------------------------------+



- Nessa temporada da Premier League de 2022-2023, a quantidade média de eventos por partida está próxima de 2500, assim como a média considerando todas as temporadas e competições.

## 1. Validação dos jogos da PL 2022-2023

In [7]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_2022_2023 = df_games.filter(col('season') == '2022-2023')

df_games_2022_2023.show(5)

+------+----------+---------+----------------------+-------------+-------------+------+--------------------+-------------+---------------+--------------+-----------------+--------------------+-------------+------------+
|gameId|      date|   season|teamExtraTimeStartSide|teamStartSide|    venueType|teamId|            teamName|competitionId|competitionName|opponentTeamId| opponentTeamName|         stadiumName|stadiumLength|stadiumWidth|
+------+----------+---------+----------------------+-------------+-------------+------+--------------------+-------------+---------------+--------------+-----------------+--------------------+-------------+------------+
|  4786|2023-05-13|2022-2023|                 Right|         Left|    TEAM_HOME|     3|         Aston Villa|            1| Premier League|            17|Tottenham Hotspur|          Villa Park|        105.0|        68.0|
|  4614|2022-12-30|2022-2023|                  Left|        Right|OPPONENT_HOME|   119|           Brentford|            

### 1.1. Mandante e Adversário

In [8]:
df_games_2022_2023.select('venueType').distinct().show()

+-------------+
|    venueType|
+-------------+
|      NEUTRAL|
|OPPONENT_HOME|
|    TEAM_HOME|
+-------------+



In [9]:
df_games_2022_2023.filter(F.col('venueType') == 'NEUTRAL').select('gameId', 'date', 'season', 'venueType', 'teamName', 'opponentTeamName', 'stadiumName').sort('date').show(truncate=False)

+------+----------+---------+---------+----------------------+-----------------------+-------------+
|gameId|date      |season   |venueType|teamName              |opponentTeamName       |stadiumName  |
+------+----------+---------+---------+----------------------+-----------------------+-------------+
|4443  |2022-08-06|2022-2023|NEUTRAL  |Chelsea               |Everton                |Goodison Park|
|4458  |2022-08-20|2022-2023|NEUTRAL  |Everton               |Nottingham Forest      |Goodison Park|
|4490  |2022-09-03|2022-2023|NEUTRAL  |Everton               |Liverpool              |Goodison Park|
|4510  |2022-09-18|2022-2023|NEUTRAL  |Everton               |West Ham               |Goodison Park|
|4531  |2022-10-09|2022-2023|NEUTRAL  |Everton               |Manchester United      |Goodison Park|
|4558  |2022-10-22|2022-2023|NEUTRAL  |Crystal Palace        |Everton                |Goodison Park|
|4578  |2022-11-05|2022-2023|NEUTRAL  |Everton               |Leicester City         |Goodi

- Apesar de Goodison Park teoricamente ser estádio do Everton e se tratarem de todos os jogos do Everton, o venueType é neutro. Por conta disso, não iremos considerar essas partidas pois é necessário saber o mandante/adversário para conseguir saber quem está com a posse na partida.
- Caso seja necessário usá-las futuramente, podemos ver como ficam os eventos de home e away e se seguem essa estrutura acima mesmo o estádio sendo neutro. Uma sugestão pode ser considerar sempre Everton como casa, mas teria que ver se os eventos de posse ficam de acordo.

### 1.2. Normalização do sentido do ataque

In [10]:
# nesse jogo o liverpool começa do lado direito atacando para a esquerda (https://www.youtube.com/watch?v=Mh82yD4YT6A)
df_games_2022_2023.filter(F.col('gameId') == 4541).show(5)

# nesse jogo o manchester city começa no lado esquerdo atacando para a direita (https://www.youtube.com/watch?v=G1JQc5F-w_g)
df_games_2022_2023.filter(F.col('gameId') == 4452).show(5)

+------+----------+---------+----------------------+-------------+---------+------+---------+-------------+---------------+--------------+----------------+-----------+-------------+------------+
|gameId|      date|   season|teamExtraTimeStartSide|teamStartSide|venueType|teamId| teamName|competitionId|competitionName|opponentTeamId|opponentTeamName|stadiumName|stadiumLength|stadiumWidth|
+------+----------+---------+----------------------+-------------+---------+------+---------+-------------+---------------+--------------+----------------+-----------+-------------+------------+
|  4541|2022-10-16|2022-2023|                 Right|        Right|TEAM_HOME|    10|Liverpool|            1| Premier League|            11| Manchester City|    Anfield|        101.0|        68.0|
+------+----------+---------+----------------------+-------------+---------+------+---------+-------------+---------------+--------------+----------------+-----------+-------------+------------+

+------+----------+-----

- A variável teamStartSide se refere sempre ao teamName. A questão é saber se esse lado se refere ao lado que o time começa a partida ou se é o sentido do ataque.

- No primeiro jogo (ID 4541), o Liverpool está jogando em casa e inicia do lado direito, ou seja, o sentido do seu ataque é para a esquerda.
- No segundo jogo (ID 4452), o AFC Bournemouth está jogando na casa do adversário Manchester City e inicia do lado direito, ou seja, o sentido do seu ataque é para a esquerda.

- Logo, teamStartSide se refere ao lado no campo que o time começa e o sentido do ataque é na direção oposta (teamStartSide = 'Right', então AttackDirection = 'Left')

In [11]:
# se venueType == TEAM_HOME, (homeTeamId == teamId e homeTeamName == teamName) e (opponentTeamId == opponentTeamId e opponentTeamName == opponentTeamName)
# se venueType == OPPONENT_HOME, (homeTeamId == opponentTeamId e homeTeamName == opponentTeamName) e (opponentTeamId == teamId e opponentTeamName == teamName)
df_games = (
    df_games_2022_2023
    .withColumns({
        # homeTeam = "team" quando o mandante é o "team" (TEAM_HOME), senão homeTeam = "opponentTeam"
        "homeTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("teamId")).otherwise(F.col("opponentTeamId")),
        "homeTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("teamName")).otherwise(F.col("opponentTeamName")),
        # homeTeamStartSide = lado que o time mandante começou:
        # - venueType == TEAM_HOME: mandante é o "team" -> usa teamStartSide direto
        # - venueType == OPPONENT_HOME: mandante é o "opponentTeam" (lado não vem direto na base) ->
        #   usa o complementar do teamStartSide (Right vira Left e vice-versa)
        "homeTeamStartSide": F.when(
            F.col("venueType") == "TEAM_HOME", F.col("teamStartSide")
            ).otherwise(
                F.when(F.col("teamStartSide") == "Right", F.lit("Left")).otherwise(F.lit("Right"))),
        
        # opponentTeam = "opponentTeam" quando o mandante é o "team" (TEAM_HOME), senão opponentTeam = "team"
        "opponentTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("opponentTeamId")).otherwise(F.col("teamId")),
        "opponentTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("opponentTeamName")).otherwise(F.col("teamName")),
        # opponentTeamStartSide = lado que o time visitante começou:
        # - venueType == TEAM_HOME: visitante é o "opponentTeam" (lado não vem direto na base) ->
        #   usa o complementar do teamStartSide
        # - venueType == OPPONENT_HOME: visitante é o próprio "team" -> usa teamStartSide direto
        "opponentTeamStartSide": F.when(
            F.col("venueType") == "TEAM_HOME", 
            F.when(F.col("teamStartSide") == "Right", F.lit("Left")).otherwise(F.lit("Right"))
            ).otherwise(F.col("teamStartSide")),
    })
    .select(
        'gameId',
        'competitionId',
        'competitionName',
        'date',
        'season',
        'venueType',
        #'homeTeamId',
        'homeTeamName',
        #'opponentTeamId',
        'opponentTeamName',
        'homeTeamStartSide',
        'opponentTeamStartSide',
        'stadiumName',
        F.col('stadiumLength').cast("float"),
        F.col('stadiumWidth').cast("float")
    )
)

df_games.show(5)

+------+-------------+---------------+----------+---------+-------------+--------------------+--------------------+-----------------+---------------------+--------------------+-------------+------------+
|gameId|competitionId|competitionName|      date|   season|    venueType|        homeTeamName|    opponentTeamName|homeTeamStartSide|opponentTeamStartSide|         stadiumName|stadiumLength|stadiumWidth|
+------+-------------+---------------+----------+---------+-------------+--------------------+--------------------+-----------------+---------------------+--------------------+-------------+------------+
|  4786|            1| Premier League|2023-05-13|2022-2023|    TEAM_HOME|         Aston Villa|   Tottenham Hotspur|             Left|                Right|          Villa Park|        105.0|        68.0|
|  4614|            1| Premier League|2022-12-30|2022-2023|OPPONENT_HOME|            West Ham|           Brentford|             Left|                Right|      London Stadium|        

In [12]:
# left join dos eventos + informações dos jogos
df_games_events = (
    df_events_2022_2023
    .join(
        df_games, 
        on = ["competitionId", "season", "gameId"],
        how='left'
    )
    # filtro para remover os jogos que tinha mandante neutro (venueType == NEUTRAL)
    .filter(~F.col('date').isNull())
)

df_games_events = (
    df_games_events
    # considerar a reversão de lado conforme mudança do primero para o segundo tempo
    # se for primeiro tempo, mantém a variável de StartSide, se não é o contrário
    .withColumns({
        'homeTeamStartSide': F.when(F.col('period') == 1, F.col('homeTeamStartSide')).otherwise(F.col('opponentTeamStartSide')),
        'opponentTeamStartSide': F.when(F.col('period') == 1, F.col('opponentTeamStartSide')).otherwise(F.col('homeTeamStartSide'))
    }) 

    # Sentido do ataque do time é sempre o lado que o outro time começou o período
    .withColumns({
        'homeTeamAttackDirection': F.col('opponentTeamStartSide'),
        'awayTeamAttackDirection': F.col('homeTeamStartSide')
    })
    #.drop('homeTeamStartSide')
)

df_games_events.show(5)

+-------------+---------+------+--------------------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+-----------+--------------+-----------------------+-----------------------+---------------+----------+-------------+--------------+----------------+-----------------+---------------------+-------------+-------------+------------+-----------------------+-----------------------+
|competitionId|   season|gameId|             eventId|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|      details_parsed|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|eventPlayerId| eventPlayerName|eventTeamId| eventTeamName|eventSubTypeDescription|eventOutcomeDescription|competitionName|      date|    venueType|  homeTeamName|opponentTeamName|homeTeamStartSide|oppone

In [13]:
filter_cols = ['gameId', 'period', 'homeTeamName', 'opponentTeamName', 'homeTeamStartSide', 'homeTeamAttackDirection', 'opponentTeamStartSide', 'awayTeamAttackDirection']

df_games_events.filter((F.col('period') == 1) & (F.col('gameId') == 4452)).select(filter_cols).show(1)
df_games_events.filter((F.col('period') == 2) & (F.col('gameId') == 4452)).select(filter_cols).show(1)

+------+------+---------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|gameId|period|   homeTeamName|opponentTeamName|homeTeamStartSide|homeTeamAttackDirection|opponentTeamStartSide|awayTeamAttackDirection|
+------+------+---------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|  4452|     1|Manchester City| AFC Bournemouth|             Left|                  Right|                Right|                   Left|
+------+------+---------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
only showing top 1 row
+------+------+---------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|gameId|period|   homeTeamName|opponentTeamName|homeTeamStartSide|homeTeamAttackDirection|opponentTeamStartSide|awayTeamAttackDirection|
+------+------+---

In [14]:
filter_cols = ['gameId', 'period', 'homeTeamName', 'opponentTeamName', 'homeTeamStartSide', 'homeTeamAttackDirection', 'opponentTeamStartSide', 'awayTeamAttackDirection']

df_games_events.filter((F.col('period') == 1) & (F.col('gameId') == 4541)).select(filter_cols).show(1)
df_games_events.filter((F.col('period') == 2) & (F.col('gameId') == 4541)).select(filter_cols).show(1)

+------+------+------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|gameId|period|homeTeamName|opponentTeamName|homeTeamStartSide|homeTeamAttackDirection|opponentTeamStartSide|awayTeamAttackDirection|
+------+------+------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|  4541|     1|   Liverpool| Manchester City|            Right|                   Left|                 Left|                  Right|
+------+------+------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
only showing top 1 row
+------+------+------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|gameId|period|homeTeamName|opponentTeamName|homeTeamStartSide|homeTeamAttackDirection|opponentTeamStartSide|awayTeamAttackDirection|
+------+------+------------+-----------

- Os times mandante e adversário, seus lados na partida e seus sentidos de ataque parecem ter sido definidos corretamente.

## 2. Validação dos eventos + informações da partida

### 2.1. Tipos de Eventos e Casos de Eventos iguais com ids diferentes

In [27]:
df_games_events.groupby('eventType', 'eventTypeDescription').count().sort('count', ascending=False).show(truncate=False)

+-------------+--------------------------------------+------+
|eventType    |eventTypeDescription                  |count |
+-------------+--------------------------------------+------+
|OTB          |A possession with a player on the ball|450276|
|PA           |Pass                                  |356680|
|CH           |Challenge                             |67118 |
|CL           |Clearance                             |17734 |
|RE           |Rebound                               |15613 |
|CR           |Cross                                 |14934 |
|SH           |Shot                                  |9959  |
|TC           |Touch Carry                           |6947  |
|BC           |Ball Carry                            |4817  |
|FIRSTKICKOFF |First half kick off                   |380   |
|SECONDKICKOFF|Second half kick off                  |380   |
|FO           |Foul                                  |316   |
+-------------+--------------------------------------+------+



In [17]:
dedup_subset_cols = [
    c for c in df_events_2022_2023.columns
    if c not in ('eventId', 'eventType', 'eventTypeDescription', 'eventSubTypeDescription', 'eventOutcomeDescription', 'details_parsed', 'homePlayers_parsed', 'awayPlayers_parsed', 'balls_parsed')
]

df_dup_groups = (
    df_events_2022_2023
    .groupBy(dedup_subset_cols)
    .agg(
        F.count('*').alias('n_duplicates'),
        F.sort_array(F.collect_set('eventType')).alias('eventTypes_in_group')
    )
    .filter(F.col('n_duplicates') > 1)
    .select('n_duplicates', 'eventTypes_in_group')
)

total_dup_groups = df_dup_groups.count()
groups_with_otb = df_dup_groups.filter(F.array_contains('eventTypes_in_group', 'OTB')).count()

print(f'Grupos de duplicata: {total_dup_groups}')
print(f'Grupos que contêm OTB: {groups_with_otb} ({groups_with_otb / total_dup_groups:.1%})')

# combinações de eventType que aparecem duplicadas juntas (ex: "OTB + PASS")
(
    df_dup_groups
    .withColumn('combo', F.concat_ws(' + ', 'eventTypes_in_group'))
    .groupBy('combo')
    .count()
    .orderBy(F.desc('count'))
    .show(50, truncate=False)
)

Grupos de duplicata: 450849
Grupos que contêm OTB: 449946 (99.8%)
+-----------------------+------+
|combo                  |count |
+-----------------------+------+
|OTB + PA               |327133|
|CH + OTB               |27924 |
|CH + OTB + PA          |25493 |
|OTB + RE               |15296 |
|CL + OTB               |14080 |
|CR + OTB               |13936 |
|OTB + SH               |7927  |
|OTB + TC               |6080  |
|CH + CL + OTB          |3603  |
|BC + OTB + PA          |2477  |
|CH + OTB + SH          |1850  |
|BC + OTB               |949   |
|CH + CR + OTB          |788   |
|CH + OTB + TC          |600   |
|BC + CH + OTB          |538   |
|BC + CH + OTB + PA     |398   |
|PA + SECONDKICKOFF     |380   |
|FIRSTKICKOFF + PA      |380   |
|BC + CR + OTB          |167   |
|OTB + PA + RE          |154   |
|FO                     |143   |
|CH + OTB + PA + TC     |133   |
|BC + OTB + SH          |107   |
|OTB + PA + TC          |68    |
|BC + CH + OTB + SH     |28    |
|CH + OTB 

- Removendo as informações de definição do evento e os dados de tracking, pode-se notar que existem eventos diferentes mas que se tratam da mesma informação. 
- OTB principalmente é o mais presente junto de outro(s) evento(s), sendo um evento redundante, pois quase 100% dos casos em que ele aparece, é duplicando um evento mais explicativo (um evento de Passe trás a mesma informação que ele e com mais detalhes). Ele basicamente é um indicativo de que o evento é uma posse. Por conta disso, vamos remover os eventos OTB para diminuir o tamanho do dataset pela metade, já que há a mesma informação em um evento válido.

- Nota-se que também existem outros tipos de eventos do mesmo momento. Para vê-los com maior facilidade, vamos remover os OTB primeiro.

In [ ]:
# ============================================================
# Remoção das duplicatas causadas pelo eventType OTB
# ============================================================

dedup_subset_cols = [
    c for c in df_events_2022_2023.columns
    if c not in ('eventId', 'eventType', 'eventTypeDescription', 'eventSubTypeDescription', 'eventOutcomeDescription', 'details_parsed', 'homePlayers_parsed', 'awayPlayers_parsed', 'balls_parsed')
]

df_events_2022_2023 = df_events_2022_2023.filter(F.col('eventType') != 'OTB')

df_dup_groups = (
    df_events_2022_2023
    .groupBy(dedup_subset_cols)
    .agg(
        F.count('*').alias('n_duplicates'),
        F.sort_array(F.collect_set('eventType')).alias('eventTypes_in_group')
    )
    .filter(F.col('n_duplicates') > 1)
    .select('n_duplicates', 'eventTypes_in_group')
)

# combinações de eventType que aparecem duplicadas juntas (ex: "CH + PA")
(
    df_dup_groups
    .withColumn('combo', F.concat_ws(' + ', 'eventTypes_in_group'))
    .groupBy('combo')
    .count()
    .orderBy(F.desc('count'))
    .show(50, truncate=False)
)

+------------------+-----+
|combo             |count|
+------------------+-----+
|CH + PA           |25493|
|CH + CL           |3603 |
|CH                |3351 |
|BC + PA           |2477 |
|CH + SH           |1850 |
|CH + CR           |788  |
|CH + TC           |600  |
|BC + CH           |538  |
|BC + CH + PA      |398  |
|PA + SECONDKICKOFF|380  |
|FIRSTKICKOFF + PA |380  |
|BC + CR           |167  |
|PA + RE           |154  |
|FO                |143  |
|CH + PA + TC      |133  |
|BC + SH           |107  |
|PA + TC           |68   |
|RE                |31   |
|BC + CH + SH      |28   |
|CH + RE           |26   |
|PA                |25   |
|CL + RE           |24   |
|RE + SH           |23   |
|CH + PA + RE      |20   |
|BC + CH + CR      |20   |
|CR + RE           |16   |
|BC + TC           |16   |
|BC                |15   |
|RE + TC           |12   |
|BC + CH + TC      |9    |
|CH + CL + TC      |6    |
|CH + SH + TC      |6    |
|CR + TC           |5    |
|SH                |5    |
|

### 2.3. Sanity da volumetria do tracking ao longo das partidas

In [19]:
df_games_events = df_games_events.withColumns({

    # Confere se nos dados de tracking do mandante não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_home":
    ((size(col("homePlayers_parsed")) > 0) & 
    forall(
        col("homePlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Confere se nos dados de tracking do adversário não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_away":
    ((size(col("awayPlayers_parsed")) > 0) & 
    forall(
        col("awayPlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    "all_balls": 
    ((size(col("balls_parsed")) > 0) & 
    forall(
        col("balls_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Traz a quantidade de dicionários de cada evento para saber se tem 11 jogadores do time mandante e adversário
    "len_tracking_home": size(col("homePlayers_parsed")),
    "len_tracking_away": size(col("awayPlayers_parsed"))
})

df_games_events.show()

+-------------+---------+------+--------------------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+-----------+--------------+-----------------------+-----------------------+---------------+----------+-------------+--------------+----------------+-----------------+---------------------+-------------+-------------+------------+-----------------------+-----------------------+-----------------+-----------------+---------+-----------------+-----------------+
|competitionId|   season|gameId|             eventId|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|      details_parsed|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|eventPlayerId| eventPlayerName|eventTeamId| eventTeamName|eventSubTypeDescription|eventOutcomeDescription|competitionName

In [20]:
df_acima_abaixo_11 = df_games_events.filter(
    ((col('len_tracking_home') != 11) & (col('len_tracking_home') > 0)) | 
    ((col('len_tracking_away') != 11) & (col('len_tracking_away') > 0)))

#df_acima_abaixo_11.cache()
df_acima_abaixo_11.show()

+-------------+---------+------+--------------------+------+-----------------+-------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+-----------+--------------------+-----------------------+-----------------------+---------------+----------+-------------+--------------------+--------------------+-----------------+---------------------+------------------+-------------+------------+-----------------------+-----------------------+-----------------+-----------------+---------+-----------------+-----------------+
|competitionId|   season|gameId|             eventId|period|periodDescription|    eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|      details_parsed|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|eventPlayerId| eventPlayerName|eventTeamId|       eventTeamName|eventSubTypeDescription|eventOutco